# V768 - Measuring Vision with Psychophysics

## Lab 7 - Discrimination vs. Estimation Part II: Motion Direction

This is the second lab in our two-part series on ***discrimination*** and ***estimation*** tasks. In the previous lab, we measured **speed** perception using random dot kinematograms. In this lab, we will measure perception of **motion direction**.

### Learning Outcomes

(i.e., what you will be able to do at the end of this lab)

- Design a discrimination experiment.

- Design an estimation experiment.

- Compare discrimination and estimation experiments, identifying strengths and weaknesses of each.

### Questions

- Include and write a caption for the LAST figure in Part A, Step 3 (Analyze the data).

- Include and write a caption for the LAST figure in Part B, Step 3 (Analyze the data).

- Reflect on your experiences in the direction discrimination and direction estimation experiments in this lab. Do you think that these experiments measure the same thing? Which of these experiments was easier for you as a subject? Why? Which of these experiments is easier to interpret as an experimenter? Why?

- The previous lab focused on speed perception and this lab focused on direction perception.  Both of these features are components of motion perception.  Reflect on your experiences in both of these labs.  Do you think that the estimation approach was more effective for measuring speed perception or direction perception?  Why?

## Part A. Direction Discrimination Experiment

### Step 1. Data collection

To collect data, open the PsychoPy ".psyexp" file in the folder titled "2d-motion-direction-discrimination", and run that experiment. The instructions will appear on the screen before the trials begin. A single run of the experiment includes 188 trials.

### Step 2. Load your data

#### Find files for specific participant ID

Before running this section of code, enter the participant ID you used below.


In [ ]:
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm

rng = np.random.default_rng(2025)


def normcdf(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    sigma = max(float(sigma), np.finfo(float).eps)
    return norm.cdf(x, loc=float(mu), scale=sigma)


def norminv(p, mu, sigma):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, np.finfo(float).eps, 1 - np.finfo(float).eps)
    sigma = max(float(sigma), np.finfo(float).eps)
    return norm.ppf(p, loc=float(mu), scale=sigma)


def psyfxn(c, x):
    c = np.asarray(c, dtype=float)
    return (normcdf(x, c[0], c[1]) * (1 - c[2] - c[3])) + c[2]


def ipsyfxn(c, p):
    c = np.asarray(c, dtype=float)
    return norminv(
        (np.asarray(p, dtype=float) - c[2]) / (1 - c[2] - c[3]),
        c[0],
        c[1],
    )


def neg_log_likelihood(params, x, k, n):
    """Binomial negative log-likelihood used by the MATLAB teaching code."""
    x = np.asarray(x, dtype=float)
    k = np.asarray(k, dtype=float)
    n = np.asarray(n, dtype=float)
    p = np.clip(psyfxn(params, x), 1e-5, 1 - 1e-5)
    return -np.sum(k * np.log(p) + (n - k) * np.log(1 - p))


def fit_psychometric_scipy(x, k, n, initial_params, bounds, method="L-BFGS-B"):
    """Return bounded maximum-likelihood point estimates using SciPy."""
    if method == "SLSQP":
        options = {"maxiter": 10_000, "ftol": 1e-12}
    else:
        options = {"maxiter": 10_000, "ftol": 1e-12, "gtol": 1e-8}
    result = minimize(
        neg_log_likelihood,
        x0=np.asarray(initial_params, dtype=float),
        args=(x, k, n),
        method=method,
        bounds=bounds,
        options=options,
    )
    if not result.success:
        raise RuntimeError(f"SciPy fit did not converge: {result.message}")
    return result


def find_participant_files(data_dir, participant_id):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            "PsychoPy data folder not found. Try changing your current Python "
            "folder to the folder containing this notebook."
        )
    file_paths = sorted(path for path in data_dir.glob(f"{participant_id}*.csv") if path.name.startswith(participant_id))
    if not file_paths:
        raise FileNotFoundError("No files found.")
    file_info = pd.DataFrame({
        "name": [path.name for path in file_paths],
        "folder": [str(path.parent) for path in file_paths],
        "path": file_paths,
    })
    print("data files found:")
    for name in file_info["name"]:
        print(f'    "{name}"')
    return file_info


def load_psychopy_csvs(file_info, remove_instruction_rows=True):
    frames = []
    for path in file_info["path"]:
        this_data = pd.read_csv(path)
        if remove_instruction_rows:
            if "thisN" in this_data.columns:
                this_data = this_data[this_data["thisN"].notna()]
            elif "trials.thisN" in this_data.columns:
                this_data = this_data[this_data["trials.thisN"].notna()]
            else:
                this_data = this_data.iloc[1:]
        frames.append(this_data.reset_index(drop=True))
    data = pd.concat(frames, ignore_index=True)
    data = data.drop(columns=["notes", "begin_experiment.started", "begin_experiment.stopped"], errors="ignore")
    return data


def display_fit_table(params, columns=("fit",)):
    return pd.DataFrame(
        np.asarray(params).reshape(4, -1),
        index=["mu", "sigma", "gamma", "lambda"],
        columns=list(columns),
    )

def circular_mean(degrees):
    radians = np.deg2rad(np.asarray(degrees, dtype=float))
    return float(np.mod(np.rad2deg(np.angle(np.sum(np.exp(1j * radians)))), 360))

participant_id = "demo"

# check current working directory and find files
discrim_file_info = find_participant_files(Path("2d-motion-direction-discrimination") / "data", participant_id)
n_files = len(discrim_file_info)


**Confirm that all of your data files are listed above.** There should be one file.

#### Load and merge data tables


In [ ]:
# load data and stack it into one table
discrim_data = load_psychopy_csvs(discrim_file_info)

# display data table size
n_trials = len(discrim_data)
print(f"number of trials = {n_trials}")


### Step 3. Analyze the data

For this experiment, the stimulus was a random dot kinematogram. You performed a **2IFC** direction discrimination task using staircase methods. In the first interval of each trial, you saw the <u>**reference**</u> stimulus, and in the second interval, you saw the <u>**test**</u> stimulus. You indicated whether the motion you saw in the second interval was ***clockwise*** or ***counterclockwise*** relative to the first interval. There were two reference directions, at 45 degrees and 135 degrees, while the test direction varied around the reference directions.

For each trial, PsychoPy saved the staircase_intensity, which contains the offset from the reference direction. The responses were stored in response as 'cw' (clockwise) or 'ccw' (counterclockwise).


In [ ]:
# stash the stimulus direction
stim_direction1 = discrim_data["staircase.base_direction"].astype(float)
stim_direction2 = stim_direction1 + discrim_data["staircase.intensity"].astype(float)

staircases = discrim_data["staircase.label"].astype(str).unique()

# convert the response key to 0 for clockwise and 1 for counterclockwise
response = discrim_data["response"].astype(str).eq("ccw")


#### Visualize the response per trial for each staircase


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=False)
ref = np.sort(discrim_data["staircase.base_direction"].astype(float).unique())

for ax, this_ref in zip(axes, ref):
    for staircase in staircases:
        stair_idx = stim_direction1.eq(this_ref) & discrim_data["staircase.label"].astype(str).eq(staircase)
        trials = np.arange(1, stair_idx.sum() + 1)
        ax.plot(trials, stim_direction1[stair_idx], 'k--')
        ax.plot(trials, stim_direction2[stair_idx], 'k')
        ax.scatter(trials[~response[stair_idx].to_numpy()], stim_direction2[stair_idx][~response[stair_idx]], marker='^', color='k')
        ax.scatter(trials[response[stair_idx].to_numpy()], stim_direction2[stair_idx][response[stair_idx]], marker='v', color='k')
    ax.set_title(f"reference direction = {this_ref:g}")
    ax.set_ylabel('direction (degrees)')
axes[-1].set_xlabel('staircase trial')
axes[0].legend(['reference direction', 'test direction', 'chose cw', 'chose ccw'], loc='best')
plt.tight_layout()
plt.show()


#### Determine probability of reporting positive (counterclockwise) difference per condition

Compute performance for each test direction (each comparison level) by calculating the probabaility of reporting that the test direction was counterclockwise relative to the reference.


In [ ]:
discrim_trials = pd.DataFrame({
    "reference_direction": stim_direction1,
    "test_direction": stim_direction2,
    "response": response,
})
discrim_results = (
    discrim_trials
    .groupby(["reference_direction", "test_direction"], as_index=False)
    .agg(n_trials=("response", "size"), mean_response=("response", "mean"))
)
display(discrim_results)


#### Fit psychometric curves to the data

Fit each reference direction separately with SciPy using the same binomial negative log-likelihood, starting values, and bounds as the MATLAB `fmincon` loop. These are maximum-likelihood point estimates with no Bayesian priors.


In [ ]:
discrim_params = np.full((len(ref), 4), np.nan)
discrim_scipy_results = []
for rr, this_ref in enumerate(ref):
    idx = discrim_results["reference_direction"].eq(this_ref)
    this_result = fit_psychometric_scipy(
        x=discrim_results.loc[idx, "test_direction"].to_numpy(dtype=float),
        k=(
            discrim_results.loc[idx, "mean_response"]
            * discrim_results.loc[idx, "n_trials"]
        ).to_numpy(dtype=float),
        n=discrim_results.loc[idx, "n_trials"].to_numpy(dtype=float),
        initial_params=[this_ref, 5, 0.1, 0.1],
        bounds=[(0, 180), (0, 100), (0, 0.5), (0, 0.5)],
    )
    discrim_scipy_results.append(this_result)
    discrim_params[rr, :] = this_result.x

discrim_param_table = pd.DataFrame(
    discrim_params.T,
    index=["mu", "sigma", "gamma", "lambda"],
    columns=[f"fit_{int(r)}" for r in ref],
)
display(discrim_param_table)
display(pd.DataFrame(
    {"negative_log_likelihood": [result.fun for result in discrim_scipy_results]},
    index=[f"fit_{int(r)}" for r in ref],
))


#### Plot the data and the psychometric curves


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

for rr, this_ref in enumerate(ref):
    # calculate curves from fits
    xx = this_ref + np.arange(-30, 31)
    yy = psyfxn(discrim_params[rr, :], xx)

    # plot the data and the fits
    idx = discrim_results["reference_direction"].eq(this_ref)
    ax.scatter(
        discrim_results.loc[idx, "test_direction"], discrim_results.loc[idx, "mean_response"],
        s=30 * discrim_results.loc[idx, "n_trials"], alpha=0.6
    )
    ax.plot(xx, yy, 'k-', linewidth=2)
    ax.axvline(this_ref, color='k', linestyle='--')
ax.set_xlabel('direction (degrees)')
ax.set_ylabel('proportion counterclockwise')
ax.legend(['fit', 'reference direction'], loc='upper center', ncol=2)
plt.show()


#### Compute and plot the PSEs and JNDs

In Lab 5 and 6, you calculated two metrics from the psychometric curve that you fit to your data. One was the **point of subjective equality (PSE)**, which tells us about the level of bias in your response, and the other was the **just noticeable difference (JND)**, which tells us something about how precisely you can distinguish between different stimulus levels.

Let's calculate those values for this experiment, and then let's visualize them.


In [ ]:
# calculate PSE
pse = np.array([float(ipsyfxn(discrim_params[rr, :], 0.5)) for rr in range(len(ref))])
discrim_param_table = pd.DataFrame({"PSE": pse, "true reference direction": ref})
display(discrim_param_table)

# calculate JND
jnd = discrim_params[:, 1] / np.sqrt(2)
print(jnd)

# plot the PSE +/- JND along with the data and fitted curve
fig, ax = plt.subplots(figsize=(9, 4.5))
for rr, this_ref in enumerate(ref):
    idx = discrim_results["reference_direction"].eq(this_ref)
    ax.scatter(discrim_results.loc[idx, "test_direction"], discrim_results.loc[idx, "mean_response"], color='k', linewidth=2)
    xx = this_ref + np.arange(-30, 31)
    yy = psyfxn(discrim_params[rr, :], xx)
    ax.plot(xx, yy, 'k-', linewidth=2)
    ax.axvline(this_ref, color='k', linestyle='--')
    ax.plot([pse[rr], pse[rr]], [0, 0.5], 'r-', linewidth=1.5)
    jnd_x = np.array([pse[rr] - jnd[rr], pse[rr] + jnd[rr], pse[rr] + jnd[rr], pse[rr] - jnd[rr]])
    jnd_y = np.array([0, 0, 0.5, 0.5])
    ax.fill(jnd_x, jnd_y, color='r', alpha=0.2)
ax.set_xlabel('direction (degrees)')
ax.set_ylabel('proportion counterclockwise')
ax.legend(["data", "fit", "reference direction", "pse", "jnd"], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=5)
plt.show()


**What do you notice?** What do you think about your results?

## Part B.  Direction Estimation Experiment

### Step 1. Data collection

To collect data, open the PsychoPy ".psyexp" file in the folder titled "2d-motion-direction-estimation", and run that experiment. The instructions will appear on the screen before the trials begin. A single run of the experiment includes 112 trials.

### Step 2. Load your data

#### Find files for specific participant ID

Before running this section of code, enter the participant ID you used below.


In [ ]:
participant_id = "demo"

# check current working directory and find files
estim_file_info = find_participant_files(Path("2d-motion-direction-estimation") / "data", participant_id)
n_files = len(estim_file_info)


**Confirm that all of your data files are listed above.** There should be one file.

#### Load and merge data tables


In [ ]:
# load data and stack it into one table
estim_data = load_psychopy_csvs(estim_file_info)

# display data table size
n_trials = len(estim_data)
print(f"number of trials = {n_trials}")


### Step 3. Analyze your data

For this experiment, the stimulus was a random dot kinematogram. The direction of the stimulus varied, and you were asked to report your estimate of the stimulus's direction using a pointer.

For each trial, PsychoPy saved the stimulus direction and your response.


In [ ]:
# stash the stimulus direction and estimation data
stim_direction = estim_data["orientation"].astype(float)
estimated_direction = np.mod(360 - estim_data["pointer_orientation"].astype(float), 360)


#### Visualize the response per trial


In [ ]:
fig, ax = plt.subplots()
ax.scatter(stim_direction, estimated_direction, color='k', linewidth=2)
ax.set_xlabel('direction (deg)')
ax.set_ylabel('direction estimate (deg)')
ax.set_xticks(np.arange(0, 361, 45))
ax.set_yticks(np.arange(0, 361, 45))
ax.set_xlim(0, 360)
ax.set_ylim(0, 360)
ax.set_aspect('equal', adjustable='box')
plt.show()


#### Determine average rating per test direction


In [ ]:
estim_trials = pd.DataFrame({"direction": stim_direction, "estimate": estimated_direction})
estim_results = (
    estim_trials
    .groupby("direction", as_index=False)
    .agg(n_trials=("estimate", "size"), circularMean_estimate=("estimate", circular_mean))
)
display(estim_results)


#### Plot average rating as a function of test direction


In [ ]:
fig, ax = plt.subplots()
ax.scatter(stim_direction, estimated_direction, color='k', linewidth=2)
ax.plot(estim_results["direction"], estim_results["circularMean_estimate"], 'r.-', markersize=20, linewidth=2)
ax.set_xlabel('direction')
ax.set_ylabel('direction estimate')
ax.set_xticks(np.arange(0, 361, 45))
ax.set_yticks(np.arange(0, 361, 45))
ax.set_xlim(0, 360)
ax.set_ylim(0, 360)
ax.set_aspect('equal', adjustable='box')
plt.show()


**What do you notice?** What do you think about your results?
